In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [4]:
data_path = "../modeldata"

In [5]:
train_df = pd.read_csv(f"{data_path}/train.csv")
test_df = pd.read_csv(f"{data_path}/test.csv")

In [6]:
print(train_df.columns)

Index(['TEAM_ABBREVIATION', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'OPP', 'OPP_PREV_WIN', 'OPP_PREV_PTS',
       'OPP_PREV_PLUSMINUS', 'WIN'],
      dtype='str')


In [7]:
# Check dataset shapes and missing targets
for name, df in zip(
    ["train_df", "test_df"], [train_df, test_df]
):
  print(f"{name} shape before dropna: {df.shape}")
  if "gf" in df.columns:
    print(f"Null 'gf' count in {name}: {df['gf'].isna().sum()}")
  else:
    print(f"WARNING: 'gf' column not found in {name}!")


print("--- After dropping NaNs ---")
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape before dropna: (46072, 11)
test_df shape before dropna: (11518, 11)
--- After dropping NaNs ---
train_df shape: (46072, 11)
test_df shape: (11518, 11)


In [8]:
#remove missiong values
train_df = train_df.dropna()
test_df = test_df.dropna()

In [9]:
target = "WIN"

In [10]:
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

In [11]:
# 3. Identify categorical and numerical columns automatically
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()
numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# 4. Create preprocessing pipelines for both data types
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

C:\Users\User\AppData\Local\Temp\ipykernel_21996\1775160886.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [12]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()
numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

C:\Users\User\AppData\Local\Temp\ipykernel_21996\2822572121.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [13]:
cat_cols = categorical_cols
num_cols = numerical_cols

# 1. Encode categorical columns to integers & calculate embedding sizes
embedding_sizes = []
X_cat_train = []
for col in cat_cols:
  le = LabelEncoder()
  encoded = le.fit_transform(train_df[col].astype(str))
  X_cat_train.append(encoded)
  num_classes = len(le.classes_)
  # Rule of thumb for embedding dimension: min(50, (num_classes + 1) // 2)
  emb_dim = min(50, (num_classes + 1) // 2)
  embedding_sizes.append((num_classes, emb_dim))

X_cat_train = np.stack(X_cat_train, axis=1)

# 2. Scale numerical columns
scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_df[num_cols])

# 3. Extract target variable
y_train = train_df["WIN"].values

In [14]:
# 1. Encode categorical columns to integers & calculate embedding sizes
embedding_sizes = []
X_cat_test = []
for col in cat_cols:
  le = LabelEncoder()
  encoded = le.fit_transform(test_df[col].astype(str))
  X_cat_test.append(encoded)
  num_classes = len(le.classes_)
  # Rule of thumb for embedding dimension: min(50, (num_classes + 1) // 2)
  emb_dim = min(50, (num_classes + 1) // 2)
  embedding_sizes.append((num_classes, emb_dim))

X_cat_test = np.stack(X_cat_test, axis=1)

# 2. Scale numerical columns
scaler = StandardScaler()
X_num_test = scaler.fit_transform(test_df[num_cols])

# 3. Extract target variable
y_test = test_df["WIN"].values

In [15]:
class MixedTabularNN(nn.Module):

  def __init__(self, embedding_sizes, num_numerical_features):
    super().__init__()
    # Embedding layers for categorical features
    self.embeddings = nn.ModuleList([
        nn.Embedding(num_classes, emb_dim)
        for num_classes, emb_dim in embedding_sizes
    ])

    total_emb_dim = sum(emb_dim for _, emb_dim in embedding_sizes)
    input_dim = total_emb_dim + num_numerical_features

    # Dense layers
    self.fc1 = nn.Linear(input_dim, 64)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(0.2)
    self.fc2 = nn.Linear(64, 32)
    self.out = nn.Linear(
        32, 1
    )  # 1 output for regression (e.g., goals prediction)

  def forward(self, x_cat, x_num):
    # Process categorical features through respective embeddings
    embedded = [
        emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)
    ]
    x_cat_concat = torch.cat(embedded, dim=1)

    # Combine categorical embeddings and numerical features
    x = torch.cat([x_cat_concat, x_num], dim=1)

    x = self.relu(self.fc1(x))
    x = self.dropout(x)
    x = self.relu(self.fc2(x))
    return self.out(x)

In [16]:
# Convert data to PyTorch tensors
tensor_cat = torch.tensor(X_cat_train, dtype=torch.long)
tensor_num = torch.tensor(X_num_train, dtype=torch.float32)
tensor_y = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

dataset = TensorDataset(tensor_cat, tensor_num, tensor_y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model, loss function, and optimizer
model = MixedTabularNN(
    embedding_sizes, num_numerical_features=len(num_cols)
)
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 10
model.train()
for epoch in range(epochs):
  epoch_loss = 0
  for x_cat_batch, x_num_batch, y_batch in dataloader:
    optimizer.zero_grad()
    predictions = model(x_cat_batch, x_num_batch)
    loss = criterion(predictions, y_batch)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()

  print(
      f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / len(dataloader):.4f}"
  )

Epoch 1/10, Loss: 0.2481
Epoch 2/10, Loss: 0.2413
Epoch 3/10, Loss: 0.2398
Epoch 4/10, Loss: 0.2391
Epoch 5/10, Loss: 0.2384
Epoch 6/10, Loss: 0.2380
Epoch 7/10, Loss: 0.2378
Epoch 8/10, Loss: 0.2373
Epoch 9/10, Loss: 0.2367
Epoch 10/10, Loss: 0.2360


In [18]:
# Convert data to PyTorch tensors
tensor_cat_test = torch.tensor(X_cat_train, dtype=torch.long)
tensor_num_test = torch.tensor(X_num_train, dtype=torch.float32)

In [19]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
import torch

model.eval()
with torch.no_grad():
  logits = model(tensor_cat_test, tensor_num_test)
  # Convert raw logits to probabilities (0 to 1) using Sigmoid
  probabilities = torch.sigmoid(logits).squeeze().numpy()

# Threshold probabilities at 0.5 to get binary predictions (0 or 1)
y_pred = (probabilities >= 0.5).astype(int)
y_true = y_test  # Actual binary labels

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

ValueError: Found input variables with inconsistent numbers of samples: [11508, 46030]